
# 03H_v2 — Temporal Dataset Quality Audit

**Notebook build ID: `03H_TEMPORAL_DATASET_QA_V2`**

This notebook is the final dataset-quality checkpoint before adding:

- HRRR weather,
- terrain,
- LANDFIRE fuels/vegetation,
- dynamic approach features,
- and machine-learning models.

The production reconstruction has already fixed the primary spatial/temporal settings:

\[
L_{\mathrm{segment}} = 250\text{ m}
\]

\[
r_{\mathrm{raster}} = 25\text{ m}
\]

\[
T_{\mathrm{primary}} = 50\text{ m}
\]

\[
G_{\mathrm{primary}} = 48\text{ h}.
\]

The purpose of 03H is not to change those choices automatically.

The purpose is to determine whether the resulting segment-level dataset is statistically and scientifically suitable for predictive modeling.

---

# Questions answered

## 1. Incident concentration

Are hundreds of thousands of segments dominated by only a few megafires?

The audit reports:

- top-1 incident share,
- top-5 share,
- top-10 share,
- top-20 share,
- effective number of incidents,
- concentration separately for Held and Burned Over.

---

## 2. Class balance across independent fires

The global class count is not sufficient.

The audit checks:

- supported Held/Burned Over counts per incident,
- incidents containing both classes,
- year balance,
- incidents potentially suitable for held-out binary evaluation.

No incident is automatically excluded simply because one class is rare.

---

## 3. Parent-line dependence

Segments are nested:

\[
\text{segment}
\subset
\text{parent fireline}
\subset
\text{incident}.
\]

The audit quantifies:

- supported segments per parent line,
- parent-line concentration,
- how many neighboring segments inherit the same outcome.

---

## 4. Temporal censoring

The production diagnostic reasons are mapped into interpretable censoring categories:

- resolved interval,
- left censored,
- right censored,
- unresolved interval/gap,
- no observed challenge,
- trajectory conflict.

This is reported separately for Held and Burned Over.

---

## 5. NIROPS temporal consistency

The production reconstruction used:

```text
USE_CUMULATIVE_NIROPS = False
```

Therefore the audit checks whether reported NIROPS fire area decreases between consecutive observations.

Area decreases can reflect perimeter revision, mapping differences, or non-monotonic operational products.

The audit flags incidents for review but does **not** automatically delete them.

---

## 6. Threshold robustness

The 03G sensitivity results are converted into interpretable comparisons.

For each class and spatial tolerance, the audit quantifies how much of the 72 h-supported cohort is already captured at 48 h.

The audit also compares spatial-tolerance behavior at the fixed 48 h primary temporal window.

---

## 7. Freeze the primary ML cohort

The primary candidate cohort is:

\[
\boxed{
\mathcal D_{\mathrm{ML}}
=
\{\text{250 m segments temporally supported at 50 m / 48 h}\}
}
\]

The output retains:

- incident ID,
- parent fireline ID,
- segment ID,
- inherited Held/Burned Over outcome,
- interval-censored challenge start/end,
- segment geometry metadata,
- later feature-join keys.

The notebook adds candidate weighting columns but does not mandate a training-weight strategy.

---

# Critical validation rule

Machine-learning validation must hold out complete incidents:

\[
\text{train IRWINIDs}
\cap
\text{test IRWINIDs}
=
\varnothing.
\]

Random segment splitting is not valid.



> **03H_v2 corrections**
>
> 1. Incident-level summaries are now keyed **only by `_irwin_norm`**. `IncidentName` and `Year` are attached afterward as metadata, preventing many-to-many row multiplication during merges.
> 2. The complete primary segment table is audited for duplicate `segment_uid` values before any statistical summaries are computed.
> 3. Duplicate physical segment IDs with identical target/evidence information are collapsed to one modeling unit.
> 4. Any duplicate segment ID with conflicting outcome/evidence/challenge interval is treated as a **critical QA conflict** and stops the notebook after writing a conflict report.
> 5. Duplicate segment IDs with only conflicting `FeatureCategory` metadata are retained once, flagged as ambiguous, and their category is set to missing rather than arbitrarily selecting a line type.
> 6. All downstream concentration, class-balance, incident metadata, and ML-cohort joins are one-row-per-IRWIN.


In [ ]:
import os  # Configurable project root for the public release.

NOTEBOOK_BUILD_ID = "03H_TEMPORAL_DATASET_QA_V2"

print("=" * 78)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 78)

from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

PROCESSED_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "fireline_dynamic_250m"
)

PRIMARY_ROOT = (
    PROCESSED_ROOT
    / "primary_50m_48h"
)

SUPPORTED_ROOT = (
    PROCESSED_ROOT
    / "temporally_supported_50m_48h"
)

SENSITIVITY_ROOT = (
    PROCESSED_ROOT
    / "sensitivity"
)

INCIDENT_META_ROOT = (
    PROCESSED_ROOT
    / "incident_metadata"
)

OUTPUT_03G_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / "nirops_temporal_production_250m"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / "temporal_dataset_quality_audit_v2"
)

ML_COHORT_ROOT = (
    PROCESSED_ROOT
    / "ml_cohort"
)

NIROPS_SEARCH_ROOT = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "joinability_audit"
    / "nirops_extracted"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

ML_COHORT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

assert PRIMARY_ROOT.exists(), PRIMARY_ROOT
assert SUPPORTED_ROOT.exists(), SUPPORTED_ROOT
assert SENSITIVITY_ROOT.exists(), SENSITIVITY_ROOT
assert INCIDENT_META_ROOT.exists(), INCIDENT_META_ROOT
assert NIROPS_SEARCH_ROOT.exists(), NIROPS_SEARCH_ROOT

PRIMARY_TOLERANCE_M = 50.0
PRIMARY_MAX_GAP_HOURS = 48.0

# Diagnostic only: these thresholds generate a review flag.
# They do NOT automatically exclude an incident from the ML cohort.
AREA_DECREASE_REL_TOL = 0.05
AREA_DECREASE_REVIEW_FRACTION = 0.10
MAX_SINGLE_DECREASE_REVIEW = 0.20

# Used only to identify incidents that can support stable per-incident
# binary evaluation metrics. This is not a training exclusion rule.
MIN_SUPPORTED_PER_CLASS_FOR_EVAL = 10

print("Primary cohort:", PRIMARY_TOLERANCE_M, "m /", PRIMARY_MAX_GAP_HOURS, "h")
print("NIROPS area-decrease tolerance:", AREA_DECREASE_REL_TOL)
print("Evaluation diagnostic minimum/class:", MIN_SUPPORTED_PER_CLASS_FOR_EVAL)


## 1. Imports

In [ ]:

import sys
import subprocess
import importlib.util

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "geopandas": "geopandas",
    "pyogrio": "pyogrio",
    "pyarrow": "pyarrow",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
}

for module, package in required.items():

    if importlib.util.find_spec(
        module
    ) is None:

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                package,
            ]
        )

import re
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 200)

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("GeoPandas:", gpd.__version__)


## 2. Verify the 03G production dataset

In [ ]:

primary_files = sorted(
    PRIMARY_ROOT.glob(
        "*.parquet"
    )
)

supported_files = sorted(
    SUPPORTED_ROOT.glob(
        "*.parquet"
    )
)

sensitivity_files = sorted(
    SENSITIVITY_ROOT.glob(
        "*.csv"
    )
)

metadata_files = sorted(
    INCIDENT_META_ROOT.glob(
        "*.json"
    )
)

print("Primary shards:", len(primary_files))
print("Supported shards:", len(supported_files))
print("Sensitivity shards:", len(sensitivity_files))
print("Incident metadata:", len(metadata_files))

assert len(primary_files) > 0
assert len(supported_files) > 0
assert len(sensitivity_files) > 0
assert len(metadata_files) > 0

if not (
    len(primary_files)
    == len(supported_files)
    == len(sensitivity_files)
    == len(metadata_files)
):

    print(
        "WARNING: shard counts differ. "
        "The notebook will audit what is present on disk."
    )



# 3. Load the primary 50 m / 48 h segment table

~644k rows is tractable in memory for this QA stage.

The segment geometry itself is not needed because the primary production shards already contain the numeric segment geometry descriptors required for later joins.


In [ ]:

primary_frames = []

for path in tqdm(
    primary_files,
    desc="Load primary 50m/48h shards",
):

    primary_frames.append(
        pd.read_parquet(
            path
        )
    )

primary_raw = pd.concat(
    primary_frames,
    ignore_index=True,
)

del primary_frames

print(
    "Raw primary segment rows:",
    f"{len(primary_raw):,}"
)

print(
    "Raw incidents:",
    f"{primary_raw['_irwin_norm'].nunique():,}"
)

print(
    "Raw parent firelines:",
    f"{primary_raw['_geom_hash'].nunique():,}"
)

# ============================================================
# PHYSICAL SEGMENT-ID DUPLICATE AUDIT
# ============================================================

duplicate_mask = (
    primary_raw[
        "segment_uid"
    ].duplicated(
        keep=False
    )
)

duplicate_rows = (
    primary_raw[
        duplicate_mask
    ]
    .copy()
)

print(
    "Rows participating in duplicate segment_uid groups:",
    f"{len(duplicate_rows):,}"
)

print(
    "Duplicate segment_uid groups:",
    f"{duplicate_rows['segment_uid'].nunique():,}"
)

def nunique_including_na(
    series,
):
    return int(
        series.nunique(
            dropna=False
        )
    )

if len(
    duplicate_rows
):

    duplicate_audit = (
        duplicate_rows
        .groupby(
            "segment_uid",
            dropna=False,
        )
        .agg(
            rows=(
                "segment_uid",
                "size",
            ),
            source_rows=(
                "source_index",
                "nunique",
            ),
            engagement_values=(
                "_engagement_norm",
                nunique_including_na,
            ),
            evidence_values=(
                "evidence",
                nunique_including_na,
            ),
            reason_values=(
                "diagnostic_reason",
                nunique_including_na,
            ),
            challenge_start_values=(
                "challenge_interval_start_utc",
                nunique_including_na,
            ),
            challenge_end_values=(
                "challenge_interval_end_utc",
                nunique_including_na,
            ),
            feature_category_values=(
                "FeatureCategory",
                nunique_including_na,
            ),
        )
        .reset_index()
    )

else:

    duplicate_audit = pd.DataFrame(
        columns=[
            "segment_uid",
            "rows",
            "source_rows",
            "engagement_values",
            "evidence_values",
            "reason_values",
            "challenge_start_values",
            "challenge_end_values",
            "feature_category_values",
        ]
    )

duplicate_audit.to_csv(
    OUTPUT_ROOT
    / "duplicate_segment_uid_audit.csv",
    index=False,
)

# Critical conflicts would mean the same physical segment ID has
# incompatible target or temporal-corroboration information.
if len(
    duplicate_audit
):

    critical_duplicate_conflicts = (
        duplicate_audit[
            (
                duplicate_audit[
                    "engagement_values"
                ] > 1
            )
            | (
                duplicate_audit[
                    "evidence_values"
                ] > 1
            )
            | (
                duplicate_audit[
                    "reason_values"
                ] > 1
            )
            | (
                duplicate_audit[
                    "challenge_start_values"
                ] > 1
            )
            | (
                duplicate_audit[
                    "challenge_end_values"
                ] > 1
            )
        ]
        .copy()
    )

else:

    critical_duplicate_conflicts = (
        duplicate_audit.copy()
    )

critical_duplicate_conflicts.to_csv(
    OUTPUT_ROOT
    / "critical_duplicate_segment_conflicts.csv",
    index=False,
)

if len(
    critical_duplicate_conflicts
):

    display(
        critical_duplicate_conflicts.head(
            50
        )
    )

    raise RuntimeError(
        "Critical duplicate segment_uid conflicts were found. "
        "See critical_duplicate_segment_conflicts.csv before proceeding."
    )

# Feature-category ambiguity is not allowed to silently choose
# one line type. Mark the physical segment as ambiguous and set
# FeatureCategory to missing after deduplication.
ambiguous_category_ids = set(
    duplicate_audit.loc[
        duplicate_audit[
            "feature_category_values"
        ] > 1,
        "segment_uid",
    ].astype(
        str
    )
)

primary_raw[
    "duplicate_feature_category_ambiguous"
] = (
    primary_raw[
        "segment_uid"
    ].astype(
        str
    ).isin(
        ambiguous_category_ids
    )
)

# Collapse redundant physical segment duplicates.
primary = (
    primary_raw
    .sort_values(
        [
            "segment_uid",
            "source_index",
        ]
    )
    .drop_duplicates(
        subset=[
            "segment_uid",
        ],
        keep="first",
    )
    .reset_index(
        drop=True
    )
)

primary.loc[
    primary[
        "duplicate_feature_category_ambiguous"
    ],
    "FeatureCategory",
] = pd.NA

print(
    "Deduplicated primary segment rows:",
    f"{len(primary):,}"
)

print(
    "Redundant physical segment rows removed:",
    f"{len(primary_raw) - len(primary):,}"
)

print(
    "Segments with ambiguous duplicate FeatureCategory:",
    f"{primary['duplicate_feature_category_ambiguous'].sum():,}"
)

assert primary[
    "segment_uid"
].is_unique

display(
    primary.head()
)


## 4. Primary evidence sanity check

In [ ]:

evidence_check = (
    primary
    .groupby(
        [
            "_engagement_norm",
            "evidence",
            "diagnostic_reason",
        ],
        dropna=False,
    )
    .agg(
        segments=(
            "segment_uid",
            "size",
        ),
        incidents=(
            "_irwin_norm",
            "nunique",
        ),
        parent_lines=(
            "_geom_hash",
            "nunique",
        ),
    )
    .reset_index()
)

evidence_check[
    "pct_within_label"
] = (
    evidence_check[
        "segments"
    ]
    / evidence_check.groupby(
        "_engagement_norm"
    )[
        "segments"
    ].transform(
        "sum"
    )
    * 100
)

display(
    evidence_check
)

evidence_check.to_csv(
    OUTPUT_ROOT
    / "primary_evidence_sanity_check.csv",
    index=False,
)



# 5. Freeze the primary supported cohort

No additional QA exclusion is applied here.

The base cohort is exactly:

```text
evidence == temporally_supported
```

under the fixed 50 m / 48 h primary definition.


In [ ]:

cohort = (
    primary[
        primary[
            "evidence"
        ]
        == "temporally_supported"
    ]
    .copy()
)

cohort[
    "hold_label"
] = np.where(
    cohort[
        "_engagement_norm"
    ]
    == "Held",
    1,
    0,
).astype(
    np.int8
)

assert cohort[
    "challenge_interval_start_utc"
].notna().all()

assert cohort[
    "challenge_interval_end_utc"
].notna().all()

assert (
    cohort[
        "challenge_interval_end_utc"
    ]
    >= cohort[
        "challenge_interval_start_utc"
    ]
).all()

assert cohort[
    "segment_uid"
].is_unique

print(
    "Primary supported cohort rows:",
    f"{len(cohort):,}"
)

print(
    "Held:",
    f"{(cohort['hold_label'] == 1).sum():,}"
)

print(
    "Burned Over:",
    f"{(cohort['hold_label'] == 0).sum():,}"
)

print(
    "Incidents represented:",
    f"{cohort['_irwin_norm'].nunique():,}"
)

print(
    "Parent lines represented:",
    f"{cohort['_geom_hash'].nunique():,}"
)



# 6. Incident concentration

Because the modeling unit is a segment, a small number of large fires can contribute many rows.

For counts \(n_i\) from incident \(i\), define the effective number of incidents:

\[
N_{\mathrm{eff}}
=
\frac{
(\sum_i n_i)^2
}{
\sum_i n_i^2
}.
\]

This is not a statistical independence estimate.

It is a simple concentration diagnostic:

- if all incidents contribute equally, \(N_{\mathrm{eff}}\) approaches the actual number of incidents;
- if a few incidents dominate, \(N_{\mathrm{eff}}\) becomes much smaller.


In [ ]:

def first_non_null(
    series,
):
    values = (
        series
        .dropna()
    )

    if len(
        values
    ):

        return values.iloc[
            0
        ]

    return pd.NA


# One metadata row per independent incident.
incident_identity = (
    cohort
    .groupby(
        "_irwin_norm",
        as_index=False,
    )
    .agg(
        IncidentName=(
            "IncidentName",
            first_non_null,
        ),
        Year=(
            "Year",
            first_non_null,
        ),
        incident_name_variants=(
            "IncidentName",
            lambda s: int(
                s.nunique(
                    dropna=False
                )
            ),
        ),
        incident_year_variants=(
            "Year",
            lambda s: int(
                s.nunique(
                    dropna=False
                )
            ),
        ),
    )
)

assert incident_identity[
    "_irwin_norm"
].is_unique

incident_class_counts_core = (
    cohort
    .groupby(
        [
            "_irwin_norm",
            "_engagement_norm",
        ],
        dropna=False,
    )
    .size()
    .rename(
        "supported_segments"
    )
    .reset_index()
)

incident_class_counts = (
    incident_class_counts_core
    .merge(
        incident_identity[
            [
                "_irwin_norm",
                "IncidentName",
                "Year",
            ]
        ],
        on="_irwin_norm",
        how="left",
        validate="many_to_one",
    )
)

incident_totals_core = (
    cohort
    .groupby(
        "_irwin_norm",
        as_index=False,
    )
    .agg(
        supported_segments=(
            "segment_uid",
            "size",
        ),
        supported_parent_lines=(
            "_geom_hash",
            "nunique",
        ),
    )
)

incident_totals = (
    incident_totals_core
    .merge(
        incident_identity[
            [
                "_irwin_norm",
                "IncidentName",
                "Year",
            ]
        ],
        on="_irwin_norm",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        "supported_segments",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

assert incident_totals[
    "_irwin_norm"
].is_unique

incident_totals[
    "share_pct"
] = (
    incident_totals[
        "supported_segments"
    ]
    / incident_totals[
        "supported_segments"
    ].sum()
    * 100
)

incident_totals[
    "cumulative_share_pct"
] = (
    incident_totals[
        "share_pct"
    ].cumsum()
)

display(
    incident_totals.head(
        25
    )
)

incident_totals.to_csv(
    OUTPUT_ROOT
    / "supported_segments_by_incident.csv",
    index=False,
)

incident_identity.to_csv(
    OUTPUT_ROOT
    / "incident_identity_audit.csv",
    index=False,
)


In [ ]:

def concentration_metrics(
    counts,
    label,
):
    values = np.asarray(
        counts,
        dtype=float,
    )

    values = values[
        values > 0
    ]

    values = np.sort(
        values
    )[
        ::-1
    ]

    total = (
        values.sum()
    )

    if (
        len(values) == 0
        or total == 0
    ):

        return {
            "group": label,
            "incidents": 0,
            "segments": 0,
            "top1_pct": np.nan,
            "top5_pct": np.nan,
            "top10_pct": np.nan,
            "top20_pct": np.nan,
            "effective_incidents": np.nan,
        }

    def top_share(
        n,
    ):
        return (
            values[
                :min(
                    n,
                    len(values)
                )
            ].sum()
            / total
            * 100
        )

    n_eff = (
        total**2
        / np.square(
            values
        ).sum()
    )

    return {
        "group": label,
        "incidents": len(
            values
        ),
        "segments": int(
            total
        ),
        "top1_pct": top_share(
            1
        ),
        "top5_pct": top_share(
            5
        ),
        "top10_pct": top_share(
            10
        ),
        "top20_pct": top_share(
            20
        ),
        "effective_incidents": n_eff,
    }


concentration_rows = [
    concentration_metrics(
        incident_totals[
            "supported_segments"
        ],
        "All supported",
    )
]

for label in [
    "Held",
    "Burned Over",
]:

    counts = (
        incident_class_counts[
            incident_class_counts[
                "_engagement_norm"
            ] == label
        ][
            "supported_segments"
        ]
    )

    concentration_rows.append(
        concentration_metrics(
            counts,
            label,
        )
    )

concentration_summary = (
    pd.DataFrame(
        concentration_rows
    )
)

display(
    concentration_summary
)

concentration_summary.to_csv(
    OUTPUT_ROOT
    / "incident_concentration_summary.csv",
    index=False,
)


## 7. Plot incident contribution distribution

In [ ]:

fig, ax = plt.subplots(
    figsize=(
        10,
        5,
    )
)

ax.plot(
    np.arange(
        1,
        len(
            incident_totals
        ) + 1
    ),
    incident_totals[
        "cumulative_share_pct"
    ].to_numpy(),
)

ax.set_xlabel(
    "Incidents ranked by supported segment count"
)

ax.set_ylabel(
    "Cumulative share of supported segments (%)"
)

ax.set_title(
    "Incident concentration in the primary ML cohort"
)

ax.grid(
    alpha=0.25
)

fig.tight_layout()

concentration_figure = (
    OUTPUT_ROOT
    / "incident_concentration_curve.png"
)

fig.savefig(
    concentration_figure,
    dpi=160,
    bbox_inches="tight",
)

plt.show()



# 8. Class balance across incidents

The binary model needs cross-incident variation.

This section distinguishes:

- all supported incidents;
- incidents containing at least one supported Held and one supported Burned Over segment;
- incidents with at least 10 supported segments from each class.

The final category is only an **evaluation diagnostic**.

It is not used to remove training data automatically.


In [ ]:

class_pivot_core = (
    incident_class_counts_core
    .pivot_table(
        index=[
            "_irwin_norm",
        ],
        columns="_engagement_norm",
        values="supported_segments",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

for column in [
    "Held",
    "Burned Over",
]:

    if column not in (
        class_pivot_core.columns
    ):

        class_pivot_core[
            column
        ] = 0

class_pivot = (
    incident_identity[
        [
            "_irwin_norm",
            "IncidentName",
            "Year",
        ]
    ]
    .merge(
        class_pivot_core,
        on="_irwin_norm",
        how="left",
        validate="one_to_one",
    )
)

for column in [
    "Held",
    "Burned Over",
]:

    class_pivot[
        column
    ] = (
        class_pivot[
            column
        ]
        .fillna(
            0
        )
        .astype(
            int
        )
    )

class_pivot[
    "supported_total"
] = (
    class_pivot[
        "Held"
    ]
    + class_pivot[
        "Burned Over"
    ]
)

class_pivot[
    "has_both_supported_classes"
] = (
    (
        class_pivot[
            "Held"
        ] > 0
    )
    & (
        class_pivot[
            "Burned Over"
        ] > 0
    )
)

class_pivot[
    "evaluation_ready_10_each"
] = (
    (
        class_pivot[
            "Held"
        ]
        >= MIN_SUPPORTED_PER_CLASS_FOR_EVAL
    )
    & (
        class_pivot[
            "Burned Over"
        ]
        >= MIN_SUPPORTED_PER_CLASS_FOR_EVAL
    )
)

class_pivot[
    "held_fraction"
] = np.where(
    class_pivot[
        "supported_total"
    ] > 0,
    class_pivot[
        "Held"
    ]
    / class_pivot[
        "supported_total"
    ],
    np.nan,
)

assert class_pivot[
    "_irwin_norm"
].is_unique

display(
    class_pivot.sort_values(
        "supported_total",
        ascending=False,
    ).head(
        30
    )
)

class_pivot.to_csv(
    OUTPUT_ROOT
    / "supported_class_balance_by_incident.csv",
    index=False,
)


In [ ]:

class_balance_summary = pd.DataFrame(
    [
        {
            "supported_incidents": class_pivot[
                "_irwin_norm"
            ].nunique(),
            "incidents_with_both_supported_classes": int(
                class_pivot[
                    "has_both_supported_classes"
                ].sum()
            ),
            "incidents_with_10plus_each_class": int(
                class_pivot[
                    "evaluation_ready_10_each"
                ].sum()
            ),
            "median_supported_segments_per_incident": class_pivot[
                "supported_total"
            ].median(),
            "median_held_fraction_per_incident": class_pivot[
                "held_fraction"
            ].median(),
        }
    ]
)

display(
    class_balance_summary
)

class_balance_summary.to_csv(
    OUTPUT_ROOT
    / "class_balance_summary.csv",
    index=False,
)


## 9. Year balance

In [ ]:

year_balance = (
    cohort
    .groupby(
        [
            "Year",
            "_engagement_norm",
        ],
        dropna=False,
    )
    .agg(
        supported_segments=(
            "segment_uid",
            "size",
        ),
        incidents=(
            "_irwin_norm",
            "nunique",
        ),
        parent_lines=(
            "_geom_hash",
            "nunique",
        ),
    )
    .reset_index()
)

year_balance[
    "pct_within_class"
] = (
    year_balance[
        "supported_segments"
    ]
    / year_balance.groupby(
        "_engagement_norm"
    )[
        "supported_segments"
    ].transform(
        "sum"
    )
    * 100
)

display(
    year_balance
)

year_balance.to_csv(
    OUTPUT_ROOT
    / "supported_class_balance_by_year.csv",
    index=False,
)



# 10. Parent-line dependence

A 250 m segmentation unit improves localization, but multiple neighboring segments from one parent fireline are not independent observations.

This section quantifies the supported-segment multiplicity of parent lines.


In [ ]:

parent_counts = (
    cohort
    .groupby(
        [
            "_irwin_norm",
            "_geom_hash",
            "_engagement_norm",
        ],
        dropna=False,
    )
    .agg(
        supported_segments=(
            "segment_uid",
            "size",
        ),
        supported_segment_length_m=(
            "actual_segment_length_m",
            "sum",
        ),
    )
    .reset_index()
)

parent_dependence_summary = pd.DataFrame(
    [
        {
            "supported_parent_lines": len(
                parent_counts
            ),
            "median_supported_segments_per_parent": parent_counts[
                "supported_segments"
            ].median(),
            "p75_supported_segments_per_parent": parent_counts[
                "supported_segments"
            ].quantile(
                0.75
            ),
            "p90_supported_segments_per_parent": parent_counts[
                "supported_segments"
            ].quantile(
                0.90
            ),
            "p95_supported_segments_per_parent": parent_counts[
                "supported_segments"
            ].quantile(
                0.95
            ),
            "max_supported_segments_per_parent": parent_counts[
                "supported_segments"
            ].max(),
            "parents_with_2plus_segments_pct": (
                (
                    parent_counts[
                        "supported_segments"
                    ] >= 2
                ).mean()
                * 100
            ),
            "parents_with_5plus_segments_pct": (
                (
                    parent_counts[
                        "supported_segments"
                    ] >= 5
                ).mean()
                * 100
            ),
        }
    ]
)

display(
    parent_dependence_summary
)

parent_dependence_summary.to_csv(
    OUTPUT_ROOT
    / "parent_line_dependence_summary.csv",
    index=False,
)

parent_counts.to_csv(
    OUTPUT_ROOT
    / "supported_segments_by_parent_line.csv",
    index=False,
)



# 11. Temporal censoring taxonomy

Production diagnostic reasons are mapped to a smaller scientific taxonomy.

The categories are descriptive; they do not change the primary evidence classification.


In [ ]:

def censoring_category(
    evidence,
    reason,
):

    if (
        evidence
        == "temporally_supported"
    ):

        return "resolved_interval"

    if reason == (
        "first_observation_already_inside"
    ):

        return "left_censored"

    if reason == (
        "challenge_at_last_observation"
    ):

        return "right_censored"

    if reason in {
        "outside_to_inside_gap_too_large",
        "approach_gap_too_large",
        "inside_without_observed_transition",
        "near_without_resolved_approach",
    }:

        return "unresolved_interval_or_gap"

    if reason == (
        "no_observed_challenge"
    ):

        return "no_observed_challenge"

    if (
        evidence
        == "trajectory_conflict"
    ):

        return "trajectory_conflict"

    return "other"


primary[
    "censoring_category"
] = [
    censoring_category(
        evidence,
        reason,
    )
    for evidence, reason in zip(
        primary[
            "evidence"
        ],
        primary[
            "diagnostic_reason"
        ],
    )
]

censoring_summary = (
    primary
    .groupby(
        [
            "_engagement_norm",
            "censoring_category",
        ],
        dropna=False,
    )
    .agg(
        segments=(
            "segment_uid",
            "size",
        ),
        incidents=(
            "_irwin_norm",
            "nunique",
        ),
    )
    .reset_index()
)

censoring_summary[
    "pct_within_label"
] = (
    censoring_summary[
        "segments"
    ]
    / censoring_summary.groupby(
        "_engagement_norm"
    )[
        "segments"
    ].transform(
        "sum"
    )
    * 100
)

display(
    censoring_summary
)

censoring_summary.to_csv(
    OUTPUT_ROOT
    / "temporal_censoring_summary.csv",
    index=False,
)



# 12. NIROPS temporal consistency audit

The NIROPS dataset is loaded directly.

The audit uses the reported `Acres` field when available because the goal here is a lightweight temporal-consistency check, not a replacement fire-area calculation.

For consecutive observations:

\[
\Delta A_{\mathrm{rel}}
=
\frac{A_{t+1}-A_t}{A_t}.
\]

A step is considered a meaningful area decrease when:

\[
\Delta A_{\mathrm{rel}} < -0.05.
\]

This threshold creates a **review flag only**.

No segment is automatically removed.


In [ ]:

def normalize_column_name(
    value,
):

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(
            value
        ).lower(),
    )


def find_column(
    columns,
    exact=(),
    contains=(),
):

    lookup = {
        normalize_column_name(
            column
        ): column
        for column in columns
    }

    for candidate in exact:

        key = normalize_column_name(
            candidate
        )

        if key in lookup:
            return lookup[
                key
            ]

    for token in contains:

        token = normalize_column_name(
            token
        )

        for key, original in (
            lookup.items()
        ):

            if token in key:
                return original

    return None


def normalize_irwin(
    value,
):

    if pd.isna(
        value
    ):
        return None

    value = re.sub(
        r"[^A-Z0-9]",
        "",
        str(
            value
        ).strip().upper(),
    )

    return (
        value
        or None
    )


nirops_candidates = list(
    NIROPS_SEARCH_ROOT.rglob(
        "NIROPS_2020_2024_R1_R6.shp"
    )
)

if not nirops_candidates:

    nirops_candidates = list(
        NIROPS_SEARCH_ROOT.rglob(
            "*.shp"
        )
    )

assert nirops_candidates

NIROPS_SHP = (
    nirops_candidates[0]
)

nirops = gpd.read_file(
    NIROPS_SHP,
    engine="pyogrio",
)

NR_IRWIN = find_column(
    nirops.columns,
    exact=[
        "IRWINID",
        "IRWIN_ID",
        "IrwinID",
    ],
    contains=[
        "irwin",
    ],
)

NR_TIME = find_column(
    nirops.columns,
    exact=[
        "UTC",
        "DateUTC",
        "DateTimeUTC",
    ],
    contains=[
        "utc",
        "dateutc",
    ],
)

NR_ACRES = find_column(
    nirops.columns,
    exact=[
        "Acres",
        "ACRES",
    ],
    contains=[
        "acres",
    ],
)

assert NR_IRWIN is not None
assert NR_TIME is not None

nirops[
    "_irwin_norm"
] = (
    nirops[
        NR_IRWIN
    ]
    .map(
        normalize_irwin
    )
)

nirops[
    "_time"
] = pd.to_datetime(
    nirops[
        NR_TIME
    ],
    errors="coerce",
    utc=True,
)

production_ids = set(
    primary[
        "_irwin_norm"
    ].unique()
)

nirops_prod = (
    nirops[
        nirops[
            "_irwin_norm"
        ].isin(
            production_ids
        )
    ]
    .copy()
)

print("NIROPS area field:", NR_ACRES)
print("Production NIROPS rows:", f"{len(nirops_prod):,}")


In [ ]:

if NR_ACRES is not None:

    area_table = (
        nirops_prod[
            [
                "_irwin_norm",
                "_time",
                NR_ACRES,
            ]
        ]
        .rename(
            columns={
                NR_ACRES: "_area_acres",
            }
        )
        .copy()
    )

    area_table[
        "_area_acres"
    ] = pd.to_numeric(
        area_table[
            "_area_acres"
        ],
        errors="coerce",
    )

else:

    print(
        "Reported Acres field not found. "
        "Computing planar area in EPSG:5070."
    )

    nirops_area = (
        nirops_prod[
            [
                "_irwin_norm",
                "_time",
                "geometry",
            ]
        ]
        .to_crs(
            "EPSG:5070"
        )
    )

    area_table = (
        nirops_area[
            [
                "_irwin_norm",
                "_time",
            ]
        ]
        .copy()
    )

    area_table[
        "_area_acres"
    ] = (
        nirops_area.geometry.area
        / 4046.8564224
    )

area_table = (
    area_table
    .dropna(
        subset=[
            "_irwin_norm",
            "_time",
            "_area_acres",
        ]
    )
    .copy()
)

# Multiple rows at the same timestamp are collapsed conservatively
# using max reported area for the incident-time diagnostic.
area_by_time = (
    area_table
    .groupby(
        [
            "_irwin_norm",
            "_time",
        ],
        as_index=False,
    )
    .agg(
        area_acres=(
            "_area_acres",
            "max",
        )
    )
    .sort_values(
        [
            "_irwin_norm",
            "_time",
        ]
    )
    .reset_index(
        drop=True
    )
)

area_by_time[
    "previous_area_acres"
] = (
    area_by_time
    .groupby(
        "_irwin_norm"
    )[
        "area_acres"
    ]
    .shift(
        1
    )
)

area_by_time[
    "relative_area_change"
] = (
    (
        area_by_time[
            "area_acres"
        ]
        - area_by_time[
            "previous_area_acres"
        ]
    )
    / area_by_time[
        "previous_area_acres"
    ]
)

area_by_time[
    "meaningful_area_decrease"
] = (
    area_by_time[
        "relative_area_change"
    ]
    < -AREA_DECREASE_REL_TOL
)

display(
    area_by_time.head(
        30
    )
)


In [ ]:

def max_shrink_fraction(
    values,
):
    values = (
        pd.Series(
            values
        )
        .dropna()
    )

    if not len(
        values
    ):
        return 0.0

    return float(
        max(
            0.0,
            -values.min(),
        )
    )


nirops_consistency = (
    area_by_time
    .groupby(
        "_irwin_norm"
    )
    .agg(
        nirops_area_observations=(
            "area_acres",
            "size",
        ),
        consecutive_area_steps=(
            "relative_area_change",
            lambda s: int(
                s.notna().sum()
            ),
        ),
        meaningful_decrease_steps=(
            "meaningful_area_decrease",
            "sum",
        ),
        max_single_area_decrease_fraction=(
            "relative_area_change",
            max_shrink_fraction,
        ),
    )
    .reset_index()
)

nirops_consistency[
    "meaningful_decrease_step_fraction"
] = np.where(
    nirops_consistency[
        "consecutive_area_steps"
    ] > 0,
    nirops_consistency[
        "meaningful_decrease_steps"
    ]
    / nirops_consistency[
        "consecutive_area_steps"
    ],
    0.0,
)

nirops_consistency[
    "temporal_consistency_review_flag"
] = (
    (
        nirops_consistency[
            "meaningful_decrease_step_fraction"
        ]
        > AREA_DECREASE_REVIEW_FRACTION
    )
    | (
        nirops_consistency[
            "max_single_area_decrease_fraction"
        ]
        > MAX_SINGLE_DECREASE_REVIEW
    )
)

display(
    nirops_consistency.sort_values(
        [
            "temporal_consistency_review_flag",
            "max_single_area_decrease_fraction",
        ],
        ascending=[
            False,
            False,
        ],
    ).head(
        40
    )
)

nirops_consistency.to_csv(
    OUTPUT_ROOT
    / "nirops_temporal_consistency_by_incident.csv",
    index=False,
)


## 13. NIROPS temporal-consistency summary

In [ ]:

nirops_consistency_summary = pd.DataFrame(
    [
        {
            "incidents_audited": len(
                nirops_consistency
            ),
            "incidents_with_any_gt5pct_area_decrease": int(
                (
                    nirops_consistency[
                        "meaningful_decrease_steps"
                    ] > 0
                ).sum()
            ),
            "incidents_flagged_for_review": int(
                nirops_consistency[
                    "temporal_consistency_review_flag"
                ].sum()
            ),
            "median_decrease_step_fraction": nirops_consistency[
                "meaningful_decrease_step_fraction"
            ].median(),
            "p90_decrease_step_fraction": nirops_consistency[
                "meaningful_decrease_step_fraction"
            ].quantile(
                0.90
            ),
            "max_single_area_decrease_fraction": nirops_consistency[
                "max_single_area_decrease_fraction"
            ].max(),
        }
    ]
)

display(
    nirops_consistency_summary
)

nirops_consistency_summary.to_csv(
    OUTPUT_ROOT
    / "nirops_temporal_consistency_summary.csv",
    index=False,
)



# 14. Load and aggregate 03G threshold sensitivity

This stage focuses on scientific interpretation rather than rerunning classification.


In [ ]:

sensitivity_frames = []

for path in tqdm(
    sensitivity_files,
    desc="Load sensitivity summaries",
):

    sensitivity_frames.append(
        pd.read_csv(
            path
        )
    )

sensitivity = pd.concat(
    sensitivity_frames,
    ignore_index=True,
)

del sensitivity_frames

sensitivity_global = (
    sensitivity
    .groupby(
        [
            "tolerance_m",
            "max_gap_hours",
            "_engagement_norm",
            "evidence",
        ],
        dropna=False,
    )
    .agg(
        segments=(
            "segments",
            "sum",
        ),
        incidents=(
            "_irwin_norm",
            "nunique",
        ),
    )
    .reset_index()
)

display(
    sensitivity_global.head(
        30
    )
)


## 15. Compact support-rate sensitivity

In [ ]:

support_sensitivity_rows = []

settings = (
    sensitivity_global[
        [
            "tolerance_m",
            "max_gap_hours",
            "_engagement_norm",
        ]
    ]
    .drop_duplicates()
)

for (
    tolerance_m,
    max_gap_hours,
    engagement_label,
) in settings.itertuples(
    index=False,
    name=None,
):

    subset = (
        sensitivity_global[
            (
                sensitivity_global[
                    "tolerance_m"
                ]
                == tolerance_m
            )
            & (
                sensitivity_global[
                    "max_gap_hours"
                ]
                == max_gap_hours
            )
            & (
                sensitivity_global[
                    "_engagement_norm"
                ]
                == engagement_label
            )
        ]
    )

    total = (
        subset[
            "segments"
        ].sum()
    )

    supported = (
        subset.loc[
            subset[
                "evidence"
            ]
            == "temporally_supported",
            "segments",
        ].sum()
    )

    support_sensitivity_rows.append(
        {
            "tolerance_m": tolerance_m,
            "max_gap_hours": max_gap_hours,
            "_engagement_norm": engagement_label,
            "segments": total,
            "supported": supported,
            "support_pct": (
                supported
                / total
                * 100
                if total
                else np.nan
            ),
        }
    )

support_sensitivity = pd.DataFrame(
    support_sensitivity_rows
)

display(
    support_sensitivity.sort_values(
        [
            "_engagement_norm",
            "tolerance_m",
            "max_gap_hours",
        ]
    )
)

support_sensitivity.to_csv(
    OUTPUT_ROOT
    / "support_rate_sensitivity.csv",
    index=False,
)



# 16. How much of the 72 h cohort is already captured at 48 h?

For each spatial tolerance and outcome class:

\[
R_{48/72}
=
\frac{
N_{\mathrm{supported}}(48h)
}{
N_{\mathrm{supported}}(72h)
}.
\]

This provides a direct interpretation of the 48 h choice.


In [ ]:

capture_rows = []

for tolerance_m in sorted(
    support_sensitivity[
        "tolerance_m"
    ].unique()
):

    for label in [
        "Held",
        "Burned Over",
    ]:

        subset = (
            support_sensitivity[
                (
                    support_sensitivity[
                        "tolerance_m"
                    ]
                    == tolerance_m
                )
                & (
                    support_sensitivity[
                        "_engagement_norm"
                    ]
                    == label
                )
            ]
        )

        count_48 = (
            subset.loc[
                subset[
                    "max_gap_hours"
                ]
                == 48.0,
                "supported",
            ].sum()
        )

        count_72 = (
            subset.loc[
                subset[
                    "max_gap_hours"
                ]
                == 72.0,
                "supported",
            ].sum()
        )

        capture_rows.append(
            {
                "tolerance_m": tolerance_m,
                "_engagement_norm": label,
                "supported_48h": count_48,
                "supported_72h": count_72,
                "capture_48_vs_72_pct": (
                    count_48
                    / count_72
                    * 100
                    if count_72
                    else np.nan
                ),
            }
        )

capture_48_vs_72 = pd.DataFrame(
    capture_rows
)

display(
    capture_48_vs_72
)

capture_48_vs_72.to_csv(
    OUTPUT_ROOT
    / "capture_48h_relative_to_72h.csv",
    index=False,
)


## 17. Spatial sensitivity at fixed 48 h

In [ ]:

spatial_48h = (
    support_sensitivity[
        support_sensitivity[
            "max_gap_hours"
        ] == 48.0
    ]
    .copy()
    .sort_values(
        [
            "_engagement_norm",
            "tolerance_m",
        ]
    )
)

display(
    spatial_48h
)

spatial_48h.to_csv(
    OUTPUT_ROOT
    / "spatial_sensitivity_at_48h.csv",
    index=False,
)



# 18. Add candidate training weights

The cohort remains segment-level, but two optional weighting schemes are saved.

## Incident-equal weight

Each incident receives equal total weight:

\[
w_{i,s}
=
\frac{1}{N_i}.
\]

## Incident-class-equal weight

Within each observed incident/class combination:

\[
w_{i,c,s}
=
\frac{1}{N_{i,c}}.
\]

Both columns are normalized to mean 1.

These are candidate modeling choices only.

The notebook does not declare either weighting scheme mandatory.


In [ ]:

incident_n = (
    cohort
    .groupby(
        "_irwin_norm"
    )[
        "segment_uid"
    ]
    .transform(
        "size"
    )
)

cohort[
    "weight_incident_equal"
] = (
    1.0
    / incident_n
)

cohort[
    "weight_incident_equal"
] = (
    cohort[
        "weight_incident_equal"
    ]
    / cohort[
        "weight_incident_equal"
    ].mean()
)

incident_class_n = (
    cohort
    .groupby(
        [
            "_irwin_norm",
            "_engagement_norm",
        ]
    )[
        "segment_uid"
    ]
    .transform(
        "size"
    )
)

cohort[
    "weight_incident_class_equal"
] = (
    1.0
    / incident_class_n
)

cohort[
    "weight_incident_class_equal"
] = (
    cohort[
        "weight_incident_class_equal"
    ]
    / cohort[
        "weight_incident_class_equal"
    ].mean()
)

display(
    cohort[
        [
            "_irwin_norm",
            "_engagement_norm",
            "weight_incident_equal",
            "weight_incident_class_equal",
        ]
    ].describe(
        include="all"
    )
)



# 19. Join QA metadata to the cohort

NIROPS temporal-consistency flags are retained as metadata.

They are **not** used as an automatic exclusion criterion.


In [ ]:

assert nirops_consistency[
    "_irwin_norm"
].is_unique

assert class_pivot[
    "_irwin_norm"
].is_unique

cohort_rows_before_qa_join = (
    len(
        cohort
    )
)

cohort = cohort.merge(
    nirops_consistency[
        [
            "_irwin_norm",
            "nirops_area_observations",
            "meaningful_decrease_steps",
            "meaningful_decrease_step_fraction",
            "max_single_area_decrease_fraction",
            "temporal_consistency_review_flag",
        ]
    ],
    on="_irwin_norm",
    how="left",
    validate="many_to_one",
)

cohort = cohort.merge(
    class_pivot[
        [
            "_irwin_norm",
            "Held",
            "Burned Over",
            "has_both_supported_classes",
            "evaluation_ready_10_each",
        ]
    ].rename(
        columns={
            "Held": "incident_supported_held_segments",
            "Burned Over": "incident_supported_burned_over_segments",
        }
    ),
    on="_irwin_norm",
    how="left",
    validate="many_to_one",
)

print(
    "Cohort rows before QA metadata joins:",
    f"{cohort_rows_before_qa_join:,}"
)

print(
    "Cohort rows after QA metadata joins:",
    f"{len(cohort):,}"
)

assert (
    len(
        cohort
    )
    == cohort_rows_before_qa_join
)

assert cohort[
    "segment_uid"
].is_unique

print(
    "PASS: QA metadata joins preserved one row per physical segment."
)



# 20. Freeze the primary ML cohort file

This file is the stable hand-off point for the feature-extraction notebooks.

No HRRR, terrain, or LANDFIRE features are added here.


In [ ]:

ML_COHORT_PATH = (
    ML_COHORT_ROOT
    / "ml_cohort_primary_250m_50m_48h_v2.parquet"
)

cohort.to_parquet(
    ML_COHORT_PATH,
    index=False,
)

print(
    "Saved primary ML cohort:"
)

print(
    ML_COHORT_PATH
)

print(
    "Rows:",
    f"{len(cohort):,}"
)

print(
    "File size:",
    f"{ML_COHORT_PATH.stat().st_size / (1024**2):,.2f} MB"
)


## 21. Freeze incident-level ML metadata

In [ ]:

incident_ml_metadata = (
    class_pivot
    .merge(
        incident_totals[
            [
                "_irwin_norm",
                "supported_parent_lines",
                "share_pct",
                "cumulative_share_pct",
            ]
        ],
        on="_irwin_norm",
        how="left",
        validate="one_to_one",
    )
    .merge(
        nirops_consistency,
        on="_irwin_norm",
        how="left",
        validate="one_to_one",
    )
)

assert incident_ml_metadata[
    "_irwin_norm"
].is_unique

INCIDENT_ML_META_PATH = (
    ML_COHORT_ROOT
    / "ml_cohort_incident_metadata_v2.csv"
)

incident_ml_metadata.to_csv(
    INCIDENT_ML_META_PATH,
    index=False,
)

display(
    incident_ml_metadata.head(
        30
    )
)



# 22. Dataset card

The dataset card freezes terminology and leakage rules before feature extraction.


In [ ]:

dataset_card = {
    "dataset_name": "Dynamic Fireline Temporal Cohort",
    "notebook_build_id": NOTEBOOK_BUILD_ID,
    "primary_spatial_unit": "250 m-or-shorter fireline segment",
    "raster_resolution_m": 25.0,
    "primary_challenge_tolerance_m": 50.0,
    "primary_max_nirops_gap_hours": 48.0,
    "outcome": {
        "hold_label_1": "Held",
        "hold_label_0": "Burned Over",
        "important_note": (
            "Outcome originates from quality-assured spatially inferred "
            "Fireline engagement labels and is retained only when NIROPS "
            "progression temporally corroborates the segment under the "
            "primary rule."
        ),
    },
    "challenge_time": (
        "Interval censored between challenge_interval_start_utc "
        "and challenge_interval_end_utc. Do not interpret as exact contact time."
    ),
    "primary_ml_cohort_rule": (
        "evidence == temporally_supported under 50 m / 48 h"
    ),
    "validation_group": "_irwin_norm",
    "validation_rule": (
        "All segments from an incident must remain in the same train/validation/test partition."
    ),
    "parent_line_key": "_geom_hash",
    "segment_key": "segment_uid",
    "candidate_weights": [
        "weight_incident_equal",
        "weight_incident_class_equal",
    ],
    "qa_flags_are_not_exclusions": [
        "temporal_consistency_review_flag",
        "has_both_supported_classes",
        "evaluation_ready_10_each",
    ],
    "next_features": [
        "HRRR interval weather",
        "wind-relative-to-line orientation",
        "NIROPS approach trajectory",
        "3DEP terrain",
        "pre-event LANDFIRE fuels/vegetation",
    ],
}

DATASET_CARD_PATH = (
    ML_COHORT_ROOT
    / "DATASET_CARD_V2.json"
)

DATASET_CARD_PATH.write_text(
    json.dumps(
        dataset_card,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        dataset_card,
        indent=2,
    )
)


## 23. Final QA decision table

In [ ]:

qa_decision_table = pd.DataFrame(
    [
        {
            "question": "Production completeness",
            "value": primary[
                "_irwin_norm"
            ].nunique(),
            "interpretation": "Number of incidents represented in the completed primary segment table.",
        },
        {
            "question": "Primary supported cohort size",
            "value": len(
                cohort
            ),
            "interpretation": "Temporally corroborated 250 m segment samples at 50 m / 48 h.",
        },
        {
            "question": "Supported incidents",
            "value": cohort[
                "_irwin_norm"
            ].nunique(),
            "interpretation": "Independent incident groups represented in the ML candidate cohort.",
        },
        {
            "question": "Incidents with both supported classes",
            "value": int(
                class_pivot[
                    "has_both_supported_classes"
                ].sum()
            ),
            "interpretation": "Fires contributing both Held and Burned Over supported segments.",
        },
        {
            "question": "Incidents with >=10 supported each class",
            "value": int(
                class_pivot[
                    "evaluation_ready_10_each"
                ].sum()
            ),
            "interpretation": "Diagnostic pool capable of per-fire binary metrics with modest sample counts.",
        },
        {
            "question": "Top-10 incident concentration (%)",
            "value": float(
                concentration_summary.loc[
                    concentration_summary[
                        "group"
                    ]
                    == "All supported",
                    "top10_pct",
                ].iloc[0]
            ),
            "interpretation": "Share of supported segment rows contributed by the ten largest incidents.",
        },
        {
            "question": "Effective incident count",
            "value": float(
                concentration_summary.loc[
                    concentration_summary[
                        "group"
                    ]
                    == "All supported",
                    "effective_incidents",
                ].iloc[0]
            ),
            "interpretation": "Concentration diagnostic; lower than the raw fire count when megafires dominate.",
        },
        {
            "question": "NIROPS temporal-consistency review flags",
            "value": int(
                nirops_consistency[
                    "temporal_consistency_review_flag"
                ].sum()
            ),
            "interpretation": "Diagnostic flags only; not automatic cohort exclusions.",
        },
    ]
)

display(
    qa_decision_table
)

qa_decision_table.to_csv(
    OUTPUT_ROOT
    / "QA_DECISION_TABLE.csv",
    index=False,
)


## 24. Write final QA report

In [ ]:

def df_to_md(
    dataframe,
    max_rows=200,
):

    if (
        dataframe is None
        or len(
            dataframe
        ) == 0
    ):

        return "_No rows._"

    try:

        return (
            dataframe
            .head(
                max_rows
            )
            .to_markdown(
                index=False
            )
        )

    except Exception:

        return (
            dataframe
            .head(
                max_rows
            )
            .to_string(
                index=False
            )
        )


report = f"""# Temporal Dataset Quality Audit — 03H_v2

Notebook build: `{NOTEBOOK_BUILD_ID}`

Generated: {datetime.now(timezone.utc).isoformat()}

## Primary cohort definition

The frozen candidate ML cohort is:

**250 m fireline segments with temporally supported Held/Burned Over outcomes under the 50 m / 48 h NIROPS corroboration rule.**

Rows: **{len(cohort):,}**

Incidents represented: **{cohort['_irwin_norm'].nunique():,}**

Parent firelines represented: **{cohort['_geom_hash'].nunique():,}**

Held segments: **{int((cohort['hold_label'] == 1).sum()):,}**

Burned Over segments: **{int((cohort['hold_label'] == 0).sum()):,}**

## Duplicate physical-segment audit

Raw primary rows: **{len(primary_raw):,}**

Deduplicated physical segment rows: **{len(primary):,}**

Redundant rows removed: **{len(primary_raw) - len(primary):,}**

Duplicate segment UID groups: **{len(duplicate_audit):,}**

Critical duplicate target/evidence conflicts: **{len(critical_duplicate_conflicts):,}**

Segments with ambiguous duplicate FeatureCategory metadata: **{int(primary['duplicate_feature_category_ambiguous'].sum()):,}**

## QA decision table

{df_to_md(qa_decision_table)}

## Incident concentration

{df_to_md(concentration_summary)}

## Class balance

{df_to_md(class_balance_summary)}

## Year balance

{df_to_md(year_balance)}

## Parent-line dependence

{df_to_md(parent_dependence_summary)}

## Temporal censoring

{df_to_md(censoring_summary)}

## NIROPS temporal consistency

{df_to_md(nirops_consistency_summary)}

## 48 h capture relative to 72 h

{df_to_md(capture_48_vs_72)}

## Spatial sensitivity at 48 h

{df_to_md(spatial_48h)}

## ML cohort hand-off

Primary cohort file:

`{ML_COHORT_PATH}`

Incident metadata:

`{INCIDENT_ML_META_PATH}`

Dataset card:

`{DATASET_CARD_PATH}`

## Statistical interpretation

The segment rows are not independent.

The hierarchy is:

incident → parent fireline → 250 m segment.

All machine-learning validation must hold out complete incidents using `_irwin_norm`.

Parent-line dependence should be retained as a secondary diagnostic.

Candidate incident-balanced weights are provided, but weighted and unweighted models should both be evaluated rather than assuming a weighting strategy in advance.

## NIROPS consistency interpretation

Area-decrease flags identify incidents whose successive NIROPS products are not strongly monotonic in reported area.

Those flags are diagnostic only.

A flagged incident should not be removed without spatial inspection or a predefined sensitivity analysis.

## Recommended next stage

If this report shows adequate cross-incident diversity and no catastrophic concentration problem, proceed to feature extraction:

1. **04A — NIROPS dynamic approach features**
2. **04B — HRRR interval weather**
3. **04C — terrain**
4. **04D — pre-event LANDFIRE fuels/vegetation**
5. baseline incident-held-out models only after those feature tables are validated.
"""

REPORT_PATH = (
    OUTPUT_ROOT
    / "TEMPORAL_DATASET_QUALITY_REPORT.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(
    report
)

print(
    "\nSaved:",
    REPORT_PATH
)


## 25. Output inventory

In [ ]:

print("=" * 78)
print("03H QA OUTPUTS")
print("=" * 78)

for path in sorted(
    OUTPUT_ROOT.iterdir()
):

    if path.is_file():

        if (
            path.stat().st_size
            >= 1024**2
        ):

            size = (
                f"{path.stat().st_size / (1024**2):,.2f} MB"
            )

        else:

            size = (
                f"{path.stat().st_size / 1024:,.1f} KB"
            )

        print(
            f"{path.name:<62}"
            f"{size:>15}"
        )

print()
print("=" * 78)
print("ML COHORT HAND-OFF")
print("=" * 78)

for path in sorted(
    ML_COHORT_ROOT.iterdir()
):

    if path.is_file():

        if (
            path.stat().st_size
            >= 1024**2
        ):

            size = (
                f"{path.stat().st_size / (1024**2):,.2f} MB"
            )

        else:

            size = (
                f"{path.stat().st_size / 1024:,.1f} KB"
            )

        print(
            f"{path.name:<62}"
            f"{size:>15}"
        )

print()
print("DONE:", NOTEBOOK_BUILD_ID)
